**Data Exploration**

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, f_oneway, pointbiserialr
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

In [20]:
# Load data
df = pd.read_csv('D:\ProjectsJSAI\IPL Winner Prediction\IPL_Winner_Model_Dataset.csv')

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (15, 12)

In [21]:
df.head()

,Match_ID,Date,Teams,Team1,Team2,Toss_Winner,Toss_Decision,team1_form_winrate_5,team2_form_winrate_5,venue_chase_winrate_prior,venue_score_prior,Match_Winner
0,335982,2008-04-18,Royal Challengers Bengaluru vs Kolkata Knight ...,Royal Challengers Bengaluru,Kolkata Knight Riders,Royal Challengers Bengaluru,field,NaN,0.4,0.5,0.0,Kolkata Knight Riders
1,335983,2008-04-19,Punjab Kings vs Chennai Super Kings,Punjab Kings,Chennai Super Kings,Chennai Super Kings,bat,NaN,0.4,0.5,0.0,Chennai Super Kings
2,335984,2008-04-19,Delhi Capitals vs Rajasthan Royals,Delhi Capitals,Rajasthan Royals,Rajasthan Royals,bat,NaN,0.4,0.5,0.0,Delhi Capitals
3,335985,2008-04-20,Mumbai Indians vs Royal Challengers Bengaluru,Mumbai Indians,Royal Challengers Bengaluru,Mumbai Indians,bat,NaN,0.4,0.5,0.0,Royal Challengers Bengaluru
4,335987,2008-04-21,Rajasthan Royals vs Punjab Kings,Rajasthan Royals,Punjab Kings,Punjab Kings,bat,NaN,0.2,0.5,0.0,Rajasthan Royals


In [22]:
df.shape

(902, 12)

In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 902 entries, 0 to 901
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Match_ID                   902 non-null    int64  
 1   Date                       902 non-null    object 
 2   Teams                      902 non-null    object 
 3   Team1                      902 non-null    object 
 4   Team2                      902 non-null    object 
 5   Toss_Winner                902 non-null    object 
 6   Toss_Decision              902 non-null    object 
 7   team1_form_winrate_5       890 non-null    float64
 8   team2_form_winrate_5       902 non-null    float64
 9   venue_chase_winrate_prior  902 non-null    float64
 10  venue_score_prior          902 non-null    float64
 11  Match_Winner               902 non-null    object 
dtypes: float64(4), int64(1), object(7)
memory usage: 84.7+ KB


In [24]:
df.isnull().sum()

Match_ID                      0
Date                          0
Teams                         0
Team1                         0
Team2                         0
Toss_Winner                   0
Toss_Decision                 0
team1_form_winrate_5         12
team2_form_winrate_5          0
venue_chase_winrate_prior     0
venue_score_prior             0
Match_Winner                  0
dtype: int64

In [32]:
df["Date"] = pd.to_datetime(df["Date"])

In [27]:
df = df.fillna(0.5)

In [28]:
df.isnull().sum()

Match_ID                     0
Date                         0
Teams                        0
Team1                        0
Team2                        0
Toss_Winner                  0
Toss_Decision                0
team1_form_winrate_5         0
team2_form_winrate_5         0
venue_chase_winrate_prior    0
venue_score_prior            0
Match_Winner                 0
dtype: int64

In [30]:
for column in df:
    unique = np.unique(df[column].fillna("0"))
    nr_values = len(unique)
    if nr_values <= 10:
        print('The number of values for feature {} :{} -- {}'.format(column, nr_values,unique))
    else:
        print('The number of values for feature {} :{} -- '.format(column, nr_values))

The number of values for feature Match_ID :902 -- 
The number of values for feature Date :720 -- 
The number of values for feature Teams :100 -- 
The number of values for feature Team1 :11 -- 
The number of values for feature Team2 :11 -- 
The number of values for feature Toss_Winner :11 -- 
The number of values for feature Toss_Decision :2 -- ['bat' 'field']
The number of values for feature team1_form_winrate_5 :11 -- 
The number of values for feature team2_form_winrate_5 :6 -- [0.  0.2 0.4 0.6 0.8 1. ]
The number of values for feature venue_chase_winrate_prior :225 -- 
The number of values for feature venue_score_prior :225 -- 
The number of values for feature Match_Winner :12 -- 


In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 902 entries, 0 to 901
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Match_ID                   902 non-null    int64         
 1   Date                       902 non-null    datetime64[ns]
 2   Teams                      902 non-null    object        
 3   Team1                      902 non-null    object        
 4   Team2                      902 non-null    object        
 5   Toss_Winner                902 non-null    object        
 6   Toss_Decision              902 non-null    object        
 7   team1_form_winrate_5       902 non-null    float64       
 8   team2_form_winrate_5       902 non-null    float64       
 9   venue_chase_winrate_prior  902 non-null    float64       
 10  venue_score_prior          902 non-null    float64       
 11  Match_Winner               902 non-null    object        
dtypes: datet

In [34]:
numerical_features = ["team1_form_winrate_5", "team2_form_winrate_5", "venue_chase_winrate_prior", "venue_score_prior"]
Categorical_Features = ["Teams", "Team1", "Team2", "Toss_Winner", "Toss_Decision","Match_Winner"]

In [36]:
print("="*70)
print("1. CORRELATION ANALYSIS")
print("="*70)

# Convert target to numeric for correlation calculations
if pd.api.types.is_numeric_dtype(df['Match_Winner']):
    target = pd.to_numeric(df['Match_Winner'], errors='coerce')
else:
    le = LabelEncoder()
    target = pd.Series(le.fit_transform(df['Match_Winner'].astype(str)), index=df.index)
    print("Encoded Match_Winner classes:", dict(zip(le.classes_, le.transform(le.classes_))))

correlation_results = []

for feature in numerical_features:
    feature_series = pd.to_numeric(df[feature], errors='coerce')
    valid_mask = feature_series.notna() & target.notna()
    x = feature_series[valid_mask]
    y = target[valid_mask]

    if len(x) < 3:
        print(f"\n{feature.upper()}: Not enough valid rows for correlation.")
        continue

    if x.nunique() <= 1 or y.nunique() <= 1:
        print(f"\n{feature.upper()}: Constant values detected; correlation undefined.")
        continue

    pearson_corr, pearson_p = pearsonr(x, y)
    spearman_corr, spearman_p = spearmanr(x, y)

    correlation_results.append({
        'Feature': feature,
        'Pearson_Corr': pearson_corr,
        'Pearson_P': pearson_p,
        'Spearman_Corr': spearman_corr,
        'Spearman_P': spearman_p
    })

    print(f"\n{feature.upper()}:")
    print(f"  Pearson:  {pearson_corr:.4f} (p={pearson_p:.2e})")
    print(f"  Spearman: {spearman_corr:.4f} (p={spearman_p:.2e})")

corr_df = pd.DataFrame(correlation_results)
if not corr_df.empty:
    corr_df = corr_df.sort_values(by='Spearman_Corr', key=lambda s: s.abs(), ascending=False)
    print("\n" + "="*70)
    print("Correlation Summary (sorted by |Spearman|)")
    print("="*70)
    display(corr_df)
else:
    print("No valid correlations could be computed.")

1. CORRELATION ANALYSIS
Encoded Match_Winner classes: {'Chennai Super Kings': 0, 'Delhi Capitals': 1, 'Draw/No Result': 2, 'Gujarat Titans': 3, 'Kolkata Knight Riders': 4, 'Lucknow Super Giants': 5, 'Mumbai Indians': 6, 'Punjab Kings': 7, 'Rajasthan Royals': 8, 'Rising Pune Supergiant': 9, 'Royal Challengers Bengaluru': 10, 'Sunrisers Hyderabad': 11}

TEAM1_FORM_WINRATE_5:
  Pearson:  -0.0261 (p=4.34e-01)
  Spearman: -0.0220 (p=5.09e-01)

TEAM2_FORM_WINRATE_5:
  Pearson:  -0.0862 (p=9.59e-03)
  Spearman: -0.0901 (p=6.78e-03)

VENUE_CHASE_WINRATE_PRIOR:
  Pearson:  0.0831 (p=1.25e-02)
  Spearman: 0.0811 (p=1.49e-02)

VENUE_SCORE_PRIOR:
  Pearson:  0.0831 (p=1.25e-02)
  Spearman: 0.0811 (p=1.49e-02)

Correlation Summary (sorted by |Spearman|)


,Feature,Pearson_Corr,Pearson_P,Spearman_Corr,Spearman_P
1,team2_form_winrate_5,-0.086211,0.009585,-0.090083,0.006785
2,venue_chase_winrate_prior,0.083099,0.012539,0.081061,0.014885
3,venue_score_prior,0.083099,0.012539,0.081061,0.014885
0,team1_form_winrate_5,-0.026103,0.433631,-0.022016,0.509003


In [37]:
print("="*70)
print("2. ANOVA TEST FOR CATEGORICAL FEATURES")
print("="*70)

# Ensure target is categorical text for grouping
target_col = 'Match_Winner'
anova_results = []

for cat_feature in Categorical_Features:
    if cat_feature == target_col:
        continue

    temp = df[[cat_feature, target_col]].copy()
    temp[cat_feature] = temp[cat_feature].astype(str)
    temp[target_col] = temp[target_col].astype(str)

    # Build groups: encoded winner values per category level
    groups = []
    category_names = []

    for category_value, group_df in temp.groupby(cat_feature):
        if len(group_df[target_col].unique()) < 2:
            continue
        local_encoder = LabelEncoder()
        encoded_winner = local_encoder.fit_transform(group_df[target_col])
        if len(encoded_winner) >= 2:
            groups.append(encoded_winner)
            category_names.append(category_value)

    if len(groups) < 2:
        print(f"\n{cat_feature}: Skipped (not enough valid groups for ANOVA)")
        continue

    f_stat, p_value = f_oneway(*groups)

    anova_results.append({
        'Feature': cat_feature,
        'Groups_Used': len(groups),
        'F_stat': f_stat,
        'P_value': p_value
    })

    print(f"\n{cat_feature.upper()}:")
    print(f"  Groups used: {len(groups)}")
    print(f"  F-statistic: {f_stat:.4f}")
    print(f"  p-value:     {p_value:.2e}")

anova_df = pd.DataFrame(anova_results)
if not anova_df.empty:
    anova_df = anova_df.sort_values(by='P_value', ascending=True).reset_index(drop=True)
    anova_df['Significant_at_0.05'] = anova_df['P_value'] < 0.05
    print("\n" + "="*70)
    print("ANOVA Summary (sorted by p-value)")
    print("="*70)
    display(anova_df)
else:
    print("No valid ANOVA results could be computed.")

2. ANOVA TEST FOR CATEGORICAL FEATURES

TEAMS:
  Groups used: 71
  F-statistic: 3.8079
  p-value:     2.49e-20

TEAM1:
  Groups used: 11
  F-statistic: 27.2143
  p-value:     1.40e-45

TEAM2:
  Groups used: 11
  F-statistic: 28.0289
  p-value:     6.82e-47

TOSS_WINNER:
  Groups used: 11
  F-statistic: 36.6402
  p-value:     2.80e-60

TOSS_DECISION:
  Groups used: 2
  F-statistic: 8.8484
  p-value:     3.01e-03

ANOVA Summary (sorted by p-value)


,Feature,Groups_Used,F_stat,P_value,Significant_at_0.05
0,Toss_Winner,11,36.640181,2.799987e-60,True
1,Team2,11,28.028920,6.819647e-47,True
2,Team1,11,27.214295,1.397515e-45,True
3,Teams,71,3.807930,2.493998e-20,True
4,Toss_Decision,2,8.848391,3.011824e-03,True
